In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

df = spark.table("workspace.ecommerce.gold_events")
display(df.limit(5))
print("Rows:", df.count())
print("Columns:", df.columns)

In [0]:
target_col = "avg_price"
numeric_cols = dict(df.dtypes)
feature_cols = [c for c in df.columns if c != target_col and numeric_cols[c] in ("int", "double")]
df_model = df.select(feature_cols + [target_col]).dropna()
print("Features:", feature_cols)
train, test = df_model.randomSplit([0.8, 0.2], seed=42)

In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml import Pipeline
import mlflow
import mlflow.spark
from mlflow.models.signature import infer_signature
from pyspark.ml.evaluation import RegressionEvaluator

# Use avg_price as the target column
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features")
lr = LinearRegression(featuresCol="features", labelCol=target_col)
ridge = LinearRegression(featuresCol="features", labelCol=target_col, regParam=0.5, elasticNetParam=0.0)
rf = RandomForestRegressor(featuresCol="features", labelCol=target_col, numTrees=50, maxDepth=8, seed=42)
models = {"LinearRegression": lr, "RidgeRegression": ridge, "RandomForest": rf}
mlflow.set_experiment("/Users/arpandasgupta56@gmail.com/gold-events-model-comparison")
evaluator = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="rmse")
results = []
for model_name, model in models.items():
    mlflow.end_run()
    with mlflow.start_run(run_name=model_name) as run:
        pipeline = Pipeline(stages=[assembler, scaler, model])
        pipeline_model = pipeline.fit(train)
        predictions = pipeline_model.transform(test)
        rmse = evaluator.evaluate(predictions)
        input_example = train.limit(5).toPandas()
        pred_sample = pipeline_model.transform(train.limit(5)).select("prediction").toPandas()
        signature = infer_signature(input_example, pred_sample)
        mlflow.log_param("model_name", model_name)
        mlflow.log_param("num_features", len(feature_cols))
        mlflow.log_metric("rmse", rmse)
        mlflow.spark.log_model(
            pipeline_model,
            artifact_path="model",
            input_example=input_example,
            signature=signature,
            dfs_tmpdir="/Volumes/workspace/ecommerce/mlflow_tmp"
        )
        results.append((model_name, rmse, run.info.run_id))
        print(f"{model_name} → RMSE: {rmse}")

In [0]:
from pyspark.sql import Row
results_df = spark.createDataFrame([Row(Model=m, RMSE=r) for m, r, _ in results])
display(results_df.orderBy("RMSE"))
best_model, best_rmse, best_run_id = min(results, key=lambda x: x[1])
mlflow.register_model(
    model_uri=f"runs:/{best_run_id}/model",
    name="workspace.ecommerce.best_price_model_gold_events"
)
print("✅ Best model registered successfully")